# Notebook 03 — Feature Engineering, Scaling and Dimensionality Reduction

**Project:** Credit Portfolio Segmentation and Risk Pattern Analysis
**Covers:** Stage 4 (*encode categorical variables, transform skewed features, scale numerical features*) and Stage 5 (*apply PCA for feature reduction and use t-SNE and UMAP to visualise borrower patterns*).

**What this notebook does**
1. Chooses the clustering features, transforms and encodes them, and scales them, with the evidence for each choice.
2. Runs **PCA** on all 1.35 million loans: how many components to keep (**PCA explained variance**, one of the project's evaluation metrics) and what each component means.
3. Draws **PCA, t-SNE and UMAP** maps of a 10,000-loan sample to show the patterns among borrowers.
4. Saves the fitted pipeline → `models/feature_pipeline.pkl` (reused by notebook 04 and the dashboard), the PCA features → `data/processed/features_pca.npy`, and the map coordinates → `data/processed/map_sample.pkl`.

**Before you run it:** notebook 01 must have been run. Select the `.venv` kernel, then **Run All**. This one takes **about 2–3 minutes**, because t-SNE and UMAP are slow (UMAP is also compiled the first time it runs).

> **Golden rule:** the clustering features never include `Default` or `id`. An automatic check in section 4.1 stops the notebook if they ever slip in.

## 0. Setup

In [ ]:
import pickle
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Make `src` importable (this notebook runs from the notebooks/ folder).
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEAN_DATA_FILE, FEATURE_PIPELINE_FILE, FIGURES_DIR, ID_COLUMN,
    MAP_SAMPLE_FILE, PCA_FEATURES_FILE, RANDOM_SEED, TARGET_COLUMN,
)
from src.dimensionality import map_labels, plot_map_panels, sample_positions
from src.eda import plot_correlation_heatmap
from src.features import (
    PCA_COMPONENTS, build_feature_pipeline, build_preprocessor,
    prepare_feature_frame,
)
from src.plot_style import (
    PRIMARY_COLOR, SURFACE, TEXT_MUTED, TEXT_PRIMARY, TEXT_SECONDARY,
    apply_style,
)

apply_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Run notebook 01 first: it creates {CLEAN_DATA_FILE}"
    )
loans = pd.read_pickle(CLEAN_DATA_FILE)
print(f"{len(loans):,} loans loaded")

---
# Stage 4 — Feature Engineering and Scaling

**Concept.** *Feature engineering* turns raw columns into inputs that describe borrowers fairly. *Scaling* puts those inputs on comparable scales.

**Why it matters.** K-Means, hierarchical clustering and DBSCAN all group borrowers by **distance**. Distance adds up the differences in every feature, so a feature measured in dollars (income: tens of thousands) would swamp one measured in points (FICO: hundreds) or percent (DTI: tens) purely because of its units. After scaling, a difference of "one standard deviation" counts the same in every feature.

## 4.1 Which columns the clustering sees

| Clustering feature | Built from | Transformation | Why |
|---|---|---|---|
| `log_income` | `revenue` | log(1 + income), then scaled | Skew 45 → 0.2; outliers 4.7% → 1.7% (notebook 01) |
| `loan_amount` | `loan_amnt` | scaled | Mild skew (0.8); a log made its outliers worse |
| `fico` | `fico_n` | scaled | Credit quality; a narrow, bounded range |
| `emp_length_years` | `emp_length` | 0–10 years; `NI` → median; scaled | Ordinal: keeps the order of the years |
| `dti` | `dti_n` | capped at the 99th percentile; 38 unknowns → median; scaled | Long tail up to 998 |
| `emp_length_unknown` | `emp_length` = `NI` | 0/1 flag, not scaled | Keeps the fact that employment was not reported |
| `home_ownership` | `home_ownership_n` | one-hot (4 columns), 0/1 | The PDF asks to encode home ownership |
| `purpose_group` | `purpose` | 14 purposes → 5 groups, one-hot, 0/1 | The PDF asks to encode purpose; 9 purposes are under 2.2% each |

**Left out on purpose:** `id` (identifier), **`Default` (the outcome)**, issue date/year/month (kept for trend analysis), `addr_state` and `zip_code` (location — profiled after clustering), `experience_c` (99.998% one value), and the raw text versions of the columns above.

`prepare_feature_frame()` in `src/features.py` builds this table, and it raises an error if `Default` or `id` is ever included.

In [ ]:
features = prepare_feature_frame(loans)

assert TARGET_COLUMN not in features.columns, "Default must never be clustered"
assert ID_COLUMN not in features.columns, "id must never be clustered"
n_rows, n_columns = features.shape
print(f"Feature table: {n_rows:,} rows × {n_columns} columns")
features.head()

**How to read the output.** One row per loan and 8 columns, the 8 features from the table. Income is now on a log scale (for example 10.92 = log(1 + \$55,000)), and purpose is already grouped. The two `assert` lines are the **proof for the viva** that `Default` and `id` are not clustering inputs.

## 4.2 Purpose groups

In [ ]:
purpose_mix = features["purpose_group"].value_counts()
pd.DataFrame({
    "loans": purpose_mix,
    "percent": (purpose_mix / len(features) * 100).round(1),
})

**How to read the output.** 14 purposes became 5 groups: **debt consolidation (58%)**, **credit card (22%)**, **personal/other (12%** — car, medical, moving, vacation, wedding, major purchase, educational, renewable energy, other), **home (7%** — home improvement and house), and **small business (1.2%)**. Small business keeps its own group because its loans behave very differently (the highest default rate in the EDA).

## 4.3 Build and fit the preprocessing pipeline

Each step is a scikit-learn building block, combined in `build_preprocessor()`:
- **`SimpleImputer(strategy="median")`** fills missing values (the 78,188 `NI` employment lengths and the 38 unknown DTIs) with the column's median. The median is not pulled by extreme values, unlike the mean.
- **`UpperCapper`** (our own small class) learns the 99th percentile of DTI and lowers anything above it to that value — **capping**, also called *winsorising*. The loan stays in the data; only its extreme value is limited.
- **`StandardScaler`** standardises each numeric column: **z = (value − mean) / standard deviation**, so each one ends up with mean 0 and standard deviation 1.
- **`OneHotEncoder`** turns each category into its own 0/1 column (RENT → `home_ownership_RENT` = 1).
- **`ColumnTransformer`** sends each group of columns through the right steps and glues the results together.

Everything is **fitted once** on the data and saved, so the dashboard can transform a new borrower with exactly the same medians, cap, means and standard deviations.

In [ ]:
preprocessor = build_preprocessor()
scaled = preprocessor.fit_transform(features)
feature_names = preprocessor.get_feature_names_out()
scaled_table = pd.DataFrame(scaled, columns=feature_names)

capper = preprocessor.named_transformers_["capped"].named_steps["cap"]
dti_cap = capper.caps_[0]
print(f"{scaled.shape[1]} columns after encoding "
      f"| DTI capped at {dti_cap:.2f}")
scaled_table.agg(["mean", "std", "min", "max"]).T.round(2)

**How to read the output.**
- The 5 numeric columns now have **mean 0.00 and standard deviation 1.00**, which is what scaling should produce. Their minimum and maximum show how many standard deviations the most extreme borrowers sit from the average.
- `dti` now stops at the cap (38.48% on the original scale), so its maximum is only about 2.4 standard deviations above the mean.
- `log_income` still reaches about **9.5**: the \$11M borrower sits 9.5 standard deviations above average even after the log. Extreme but real borrowers like this are exactly what notebook 04's anomaly detection looks for.
- The flag and one-hot columns are still **0/1**: their means are simply shares (for example, `home_ownership_MORTGAGE` mean 0.49 = 49% of loans).
- 8 features became **15 columns**: 5 numeric + 1 flag + 4 home-ownership + 5 purpose-group columns.

## 4.4 Why the 0/1 columns are not standardised

Standardising a rare category produces huge numbers. Here is the calculation for home ownership OTHER (517 loans):

In [ ]:
share_other = (features["home_ownership"] == "OTHER").mean()
z_of_a_one = (1 - share_other) / np.sqrt(share_other * (1 - share_other))
print(f"Share of loans with OTHER: {share_other:.4%}")
print(f"A standardised 1 for OTHER would become {z_of_a_one:.1f}")

**How to read the output.** A borrower with OTHER would sit about **51 standard deviations** away on that one column. Most numeric values lie between −3 and +3, so those 517 loans would dominate every distance, and clustering would mainly separate them from everyone else. Keeping 0/1 columns on their natural 0–1 scale keeps every category **visible but not overpowering**. This is a design decision worth explaining in the viva. Notebook 04 also tests the alternative (scaling everything) so you can compare the results.

---
# Stage 5 — Dimensionality Reduction

**Concept.**
- **PCA (Principal Component Analysis)** creates new axes, called *components*, that are weighted combinations of the original columns. Component 1 points in the direction where borrowers differ most, component 2 in the next-largest direction at a right angle to the first, and so on. Keeping the first few components keeps most of the information in fewer columns. It is **linear** and is used here for **feature reduction**.
- **t-SNE** and **UMAP** are **non-linear** methods that squeeze many columns into a 2-D picture while keeping *neighbours close together*. They are **only for visualisation**: their axes have no meaning, and the distance between two far-apart islands should not be read literally.

**Why it matters.** PCA removes redundancy (overlapping columns) and noise before clustering. The 2-D maps let us *see* whether borrowers form natural groups — the project's "PCA, t-SNE and UMAP plots" deliverable.

## 5.1 PCA on all loans — how many components?

The **explained variance ratio** of a component is the share of the total variation it captures. We keep the smallest number of components that together explain **at least 90%**.

In [ ]:
pca_full = PCA(random_state=RANDOM_SEED).fit(scaled)
ratios = pca_full.explained_variance_ratio_
cumulative = ratios.cumsum()
n_keep = int(np.argmax(cumulative >= 0.90) + 1)

variance_table = pd.DataFrame({
    "explained_%": (ratios * 100).round(2),
    "cumulative_%": (cumulative * 100).round(1),
}, index=[f"PC{i}" for i in range(1, len(ratios) + 1)])
print(f"Components needed for 90%: {n_keep} "
      f"(they explain {cumulative[n_keep - 1]:.1%})")
variance_table.T

In [ ]:
labels = variance_table.index
bar_colors = [PRIMARY_COLOR if i < n_keep else TEXT_MUTED
              for i in range(len(labels))]

fig, ax = plt.subplots(figsize=(11, 4.6))
ax.bar(labels, variance_table["explained_%"], width=0.6, color=bar_colors)
ax.plot(labels, variance_table["cumulative_%"], color=TEXT_SECONDARY,
        marker="o", markersize=6, markeredgecolor=SURFACE,
        markeredgewidth=2)
ax.axhline(90, color=TEXT_PRIMARY, linewidth=1)
ax.text(len(labels) - 1, 91.5, "90% threshold", ha="right", fontsize=8,
        color=TEXT_SECONDARY)
ax.annotate(
    f"{n_keep} components keep {cumulative[n_keep - 1]:.1%}",
    xy=(n_keep - 1, cumulative[n_keep - 1] * 100),
    xytext=(n_keep + 0.6, 66), fontsize=9, color=TEXT_PRIMARY,
    arrowprops={"arrowstyle": "-", "color": TEXT_SECONDARY, "lw": 0.8},
)
ax.set_title("PCA explained variance: bars = each component, "
             "line = running total")
ax.set_ylabel("Variance explained (%)")
ax.set_ylim(0, 105)
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_pca_explained_variance.png")
plt.show()

**How to read the chart.** Blue bars are the components we keep, grey bars the ones we drop, and the line is the running total.
- PC1 explains about **26.7%**, PC2 **17.5%**, PC3 and PC4 about **15% each**. After that the bars fall quickly.
- **7 components explain 93.1%** — the first count above the 90% threshold — so 15 columns shrink to 7 while keeping 93% of the information.
- The last two components explain **exactly 0%**. The 4 home-ownership columns always add up to 1, and so do the 5 purpose columns, so one column in each group is redundant. PCA spots this automatically.

## 5.2 What do the 7 components mean?

The **loadings** show how much each original column contributes to each component, from −1 to +1. Large values in either direction matter.

In [ ]:
pca_7 = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED).fit(scaled)
loadings = pd.DataFrame(
    pca_7.components_.T, index=feature_names,
    columns=[f"PC{i}" for i in range(1, PCA_COMPONENTS + 1)],
)

fig, ax = plt.subplots(figsize=(9, 8))
image = plot_correlation_heatmap(loadings, ax,
                                 "PCA loadings (first 7 components)")
fig.colorbar(image, ax=ax, shrink=0.6)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_pca_loadings.png")
plt.show()

**How to read the heatmap** (red = positive, blue = negative, grey = close to 0). Read one column at a time:
- **PC1 — financial capacity:** income and loan amount together (plus some FICO and employment). Borrowers high on PC1 earn more and borrow more.
- **PC2 — debt burden:** mostly DTI, with longer employment and lower FICO.
- **PC3 — stability and credit quality:** employment years and FICO, against loan size.
- **PC4:** FICO and DTI against employment years.
- **PC5 — affordability:** loan amount *against* income, i.e. a large loan for one's income.
- **PC6 — housing:** mortgage (+) against rent (−).
- **PC7 — loan purpose:** debt consolidation (+) against credit card (−).

⚠️ **Limitation to know:** the rare 0/1 columns — `emp_length_unknown`, OWN, OTHER, home, personal/other, small business — are almost grey in all 7 components. PCA keeps the directions with the most variation, and rare categories carry little of it, so they are mostly left out of the clustering input. We still use them **after** clustering to describe each segment.

## 5.3 Fit and save the final pipeline (preprocessing + PCA with 7 components)

One `Pipeline` object holds every step from raw features to the 7 PCA columns. We save it with **pickle**, so notebook 04 and the dashboard can reuse it unchanged.

In [ ]:
assert n_keep == PCA_COMPONENTS, "Update PCA_COMPONENTS in src/features.py"

feature_pipeline = build_feature_pipeline(n_components=PCA_COMPONENTS)
features_pca = feature_pipeline.fit_transform(features).astype(np.float32)

FEATURE_PIPELINE_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(FEATURE_PIPELINE_FILE, "wb") as file:
    pickle.dump(feature_pipeline, file)
np.save(PCA_FEATURES_FILE, features_pca)

kept = feature_pipeline.named_steps["pca"].explained_variance_ratio_.sum()
print(f"PCA features: {features_pca.shape} | variance kept {kept:.1%}")
print(f"Pipeline saved to {FEATURE_PIPELINE_FILE.name} "
      f"({FEATURE_PIPELINE_FILE.stat().st_size / 1024:.0f} KB)")

**How to read the output.** All 1,347,681 loans are now described by **7 numbers each** (the PCA features, saved as `float32` to halve the file size). The pipeline file is tiny, a few KB, because it stores only the learned settings (medians, the DTI cap, means, standard deviations and the PCA directions), not the data.

## 5.4 A 10,000-loan sample for the maps

t-SNE and UMAP compare each borrower with many others, so their running time grows quickly with the number of rows: 1.35 million loans would take hours. A **random sample of 10,000** (fixed seed, so the same loans every run) is plenty to see the patterns.

In [ ]:
positions = sample_positions(len(loans))
sample_pca = features_pca[positions]
labels = map_labels(loans.iloc[positions])
print(f"Sample: {len(positions):,} loans")
labels["home"].value_counts()

## 5.5 PCA map

The simplest 2-D view: each loan placed at its first two PCA components. We colour the **same map three times** — by home ownership, loan purpose and FICO band — to see which characteristics organise it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
plot_map_panels(axes, sample_pca[:, :2], labels, "PCA")
for ax in axes:  # unlike t-SNE/UMAP, PCA axes have a meaning
    ax.set_xlabel("PC1: financial capacity →")
    ax.set_ylabel("PC2: debt burden →")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_pca_map.png")
plt.show()

**How to read the maps.** The PCA map is **one big cloud** with no separate islands.
- **Home ownership:** renters lean to the left and mortgage holders to the right (they tend to earn and borrow more, which is what PC1 measures), but the two groups overlap heavily.
- **Purpose:** the three groups overlap completely, because purpose lives in PC7, not in PC1 or PC2.
- **FICO:** the shading changes gradually, from lighter (lower scores) at the top left to darker (higher scores) at the bottom right, with no clean boundary.

This already hints that borrowers form a **continuum** rather than tidy, separate groups. Keep that in mind when you see the silhouette scores in notebook 04.

## 5.6 t-SNE map

`perplexity=30` roughly sets how many neighbours each point pays attention to (5–50 is typical). `init="pca"` starts from the PCA layout, which makes the result more stable, and `random_state` fixes the randomness.

In [ ]:
start = time.time()
tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto",
            random_state=RANDOM_SEED)
tsne_map = tsne.fit_transform(sample_pca)
print(f"t-SNE finished in {time.time() - start:.0f} s")

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
plot_map_panels(axes, tsne_map, labels, "t-SNE")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_tsne_map.png")
plt.show()

**How to read the maps.** t-SNE reveals structure that PCA's first two components hid:
- **Home ownership splits the map into two halves:** renters on one side, mortgage holders on the other, with outright owners scattered across both.
- **Purpose splits it again:** debt consolidation loans and credit-card loans occupy different regions.
- **FICO does *not* form its own islands.** Higher scores (dark) gather in a band through the middle of each half and lower scores (light) sit around the edges, so credit quality varies *within* the regions formed by housing and purpose.

**Careful:** in t-SNE the size of a region and the gap between regions are not meaningful. Only "who sits next to whom" is.

## 5.7 UMAP map

UMAP (from the `umap-learn` package) has the same goal as t-SNE but is usually faster and keeps more of the overall layout. `n_neighbors=15` is how many neighbours it looks at, and `min_dist=0.1` is how tightly it packs points. With a fixed `random_state`, UMAP runs on one CPU core and may print a warning saying so — that is expected and harmless.

In [ ]:
import umap  # installed as the umap-learn package

start = time.time()
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                       random_state=RANDOM_SEED)
umap_map = umap_model.fit_transform(sample_pca)
print(f"UMAP finished in {time.time() - start:.0f} s")

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
plot_map_panels(axes, umap_map, labels, "UMAP")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_umap_map.png")
plt.show()

**How to read the maps — your turn.** Compare the UMAP maps with the t-SNE maps above, and answer these in your own words (this is this notebook's exercise):
1. Do renters and mortgage holders again form separate regions?
2. Do debt-consolidation and credit-card loans separate the same way?
3. Does FICO form its own islands, or vary inside the regions as in t-SNE?

If your answers match t-SNE, two different methods agree on the same structure — strong evidence that it is real and not an accident of one method.

## 5.8 Save the map coordinates for notebook 04

Notebook 04 colours these same maps by **cluster**, so we save the coordinates instead of re-running t-SNE and UMAP.

In [ ]:
map_sample = pd.DataFrame({
    "position": positions,
    "pc1": sample_pca[:, 0], "pc2": sample_pca[:, 1],
    "tsne_1": tsne_map[:, 0], "tsne_2": tsne_map[:, 1],
    "umap_1": umap_map[:, 0], "umap_2": umap_map[:, 1],
})
map_sample.to_pickle(MAP_SAMPLE_FILE)
print(f"Saved {len(map_sample):,} map positions to {MAP_SAMPLE_FILE.name}")

## Summary — Stages 4 and 5

| Step | Result |
|---|---|
| Clustering features | 8 features → 15 columns: 5 scaled numeric + 1 flag + 9 one-hot |
| Transformations | log(income); DTI capped at 38.48; medians for 78,188 + 38 missing values; standard scaling |
| Not clustered | `Default`, `id`, dates, state, zip, `experience_c` (an `assert` proves the first two) |
| PCA | **7 components keep 93.1%** of the variance (threshold 90%) |
| Component meanings | capacity, debt burden, stability/credit, …, affordability, housing, purpose |
| Maps | PCA = one cloud; t-SNE (and UMAP — check yours) = regions by housing and purpose, FICO varies inside them |
| Saved | `models/feature_pipeline.pkl`, `data/processed/features_pca.npy`, `data/processed/map_sample.pkl`, 5 figures |

**Next:** Notebook 04 — K-Means, hierarchical clustering and DBSCAN on these 7 PCA features (Stages 6–7).

---
## Viva practice — Stages 4 and 5

**1. Why did you scale the features?**
Clustering uses distances, and unscaled features in bigger units (dollars) would dominate those in smaller units (FICO points, DTI %). Standardising gives every numeric feature mean 0 and standard deviation 1, so each counts equally.

**2. Why StandardScaler and not MinMaxScaler?**
MinMax squeezes each feature into 0–1 using its minimum and maximum, so one extreme value (an \$11M income) would compress everyone else into a tiny range. Standardising, after the log transform and the DTI cap, is far less sensitive to extremes.

**3. Why log-transform only income?**
Evidence: the log cut income's skew from 45 to 0.2 and its outliers from 4.7% to 1.7%, but it *increased* the outliers for DTI and loan amount. Transformations are chosen per column.

**4. What is capping (winsorising), and why for DTI?**
Values above a chosen percentile are set to that percentile. DTI has a tail up to 998, so capping at the 99th percentile (38.48) keeps every loan but stops a few extreme ratios from stretching the scale.

**5. Why are the one-hot columns not standardised?**
A rare category would explode: a 1 for home ownership OTHER would become about 51 standard deviations and dominate every distance. Keeping 0/1 keeps categories visible but not overpowering.

**6. What is PCA, and why 7 components?**
PCA builds new axes along the directions of greatest variation. Seven components keep 93.1% of the variance, the first count above our 90% threshold, while removing redundant columns (the last two components explain exactly 0%).

**7. What does PC1 represent?**
Financial capacity: income and loan amount load strongly together, so borrowers high on PC1 earn more and borrow more.

**8. What is the difference between PCA, t-SNE and UMAP?**
PCA is linear and keeps global variance, so it can be used for feature reduction. t-SNE and UMAP are non-linear and keep local neighbourhoods; they are for visualisation only, and their axes and gaps have no meaning. UMAP is usually faster and keeps more of the global layout.

**9. Why run t-SNE and UMAP on a sample?**
Their cost grows quickly with the number of points: 1.35 million loans would take hours. A fixed-seed random sample of 10,000 shows the same patterns in about a minute.

**10. Is `Default` used anywhere in these stages?**
No. `prepare_feature_frame()` builds the features without it, and two `assert` statements stop the notebook if `Default` or `id` ever appear in the feature table.